# 1. What you'll learn

        - compare XGBoost, LightGBM, and CatBoost on one split
- read validation loss and stop before overfitting
- test a correct and incorrect monotonic constraint

        **The one question this notebook answers:** How do three boosting libraries repair errors sequentially, and how do early stopping and monotonic constraints control them?

# 2. Setup

This lesson keeps gradient boosting: xgboost, lightgbm, and catboost small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score,balanced_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier,early_stopping,record_evaluation
from catboost import CatBoostClassifier

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The breast-cancer dataset supplies 30 numeric predictors and a malignant target. A separate validation slice controls boosting rounds; the test remains untouched.

**Small example:** If five residual-like errors are [0.6,-0.2,0.5,-0.1,0.4], a small tree may predict +0.5 for one region and -0.15 for another; learning rate 0.1 adds only one tenth of that correction.

In [ ]:
b=load_breast_cancer(as_frame=True);X=b.data;y=(b.target==0).astype(int);print("Shape:",X.shape,"malignant share:",round(y.mean(),3));print(X.iloc[:3,:5].round(2))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));y.value_counts().plot.bar(ax=axes[0],color="#176b66");axes[0].set_title("Class counts")
sns.histplot(x=X["worst radius"],hue=y,element="step",stat="density",common_norm=False,ax=axes[1]);axes[1].set_title("Strong nonlinear threshold signal")
sns.heatmap(X.iloc[:,:12].corr(),cmap="vlag",center=0,xticklabels=False,yticklabels=False,ax=axes[2]);axes[2].set_title("Correlated tabular inputs");plt.tight_layout();plt.show()

**Takeaway:** The minority class makes stratified splits important.

**Takeaway:** A single threshold helps but overlap requires multiple corrections and interactions.

**Takeaway:** Correlated features can share importance and make explanations non-unique.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED);X_train,X_val,y_train,y_val=train_test_split(X_dev,y_dev,test_size=.25,stratify=y_dev,random_state=SEED);print("Train/validation/test:",X_train.shape,X_val.shape,X_test.shape)

# 6. Train

        Training turns the assumptions behind gradient boosting: xgboost, lightgbm, and catboost into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Begin with a constant log-odds prediction.
2. Compute each row's current loss gradient.
3. Fit a shallow tree to useful gradient regions.
4. Add a learning-rate-scaled correction and stop when validation loss stalls.

In [ ]:
xgb=XGBClassifier(n_estimators=160,max_depth=3,learning_rate=.05,subsample=.85,colsample_bytree=.85,eval_metric="logloss",random_state=SEED,n_jobs=1)
xgb.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=False)
lgb_hist={};lgb=LGBMClassifier(n_estimators=300,num_leaves=15,learning_rate=.04,verbosity=-1,random_state=SEED,n_jobs=1)
lgb.fit(X_train,y_train,eval_set=[(X_val,y_val)],callbacks=[early_stopping(20,verbose=False),record_evaluation(lgb_hist)])
cat=CatBoostClassifier(iterations=250,depth=4,learning_rate=.04,loss_function="Logloss",random_seed=SEED,verbose=False,thread_count=1,allow_writing_files=False)
cat.fit(X_train,y_train,eval_set=(X_val,y_val),early_stopping_rounds=20,verbose=False)
models={"XGBoost":xgb,"LightGBM":lgb,"CatBoost":cat}
for name,m in models.items(): print(name,"test AUC",round(roc_auc_score(y_test,m.predict_proba(X_test)[:,1]),3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
aucs={n:roc_auc_score(y_test,m.predict_proba(X_test)[:,1]) for n,m in models.items()};base=.5;xhist=xgb.evals_result()["validation_0"]["logloss"];chist=cat.get_evals_result()["validation"]["Logloss"];lhist=lgb_hist["valid_0"]["binary_logloss"]
fig,axes=plt.subplots(1,3,figsize=(15,3.8));axes[0].bar(list(aucs),list(aucs.values()),color=["#176b66","#d38b45","#768f8b"]);axes[0].axhline(base,ls="--",color="black");axes[0].set_ylim(.45,1);axes[0].set_title("Held-out ROC AUC")
axes[1].plot(xhist,label="XGBoost");axes[1].plot(lhist,label="LightGBM");axes[1].plot(chist,label="CatBoost");axes[1].set(title="Validation log loss",xlabel="boosting round");axes[1].legend()
imp=pd.Series(xgb.feature_importances_,index=X.columns).nlargest(8).sort_values();axes[2].barh(imp.index,imp.values,color="#176b66");axes[2].set_title("XGBoost split importance");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print("Random-ranking AUC baseline=0.500")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Sequential fitting can chase validation noise; a monotonic constraint with the wrong direction blocks the real relationship. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
feature="worst radius";mono_good={c:(1 if c==feature else 0) for c in X.columns};mono_bad={c:(-1 if c==feature else 0) for c in X.columns}
good=XGBClassifier(n_estimators=100,max_depth=3,learning_rate=.06,monotone_constraints=mono_good,random_state=SEED,n_jobs=1).fit(X_train,y_train);bad=XGBClassifier(n_estimators=100,max_depth=3,learning_rate=.06,monotone_constraints=mono_bad,random_state=SEED,n_jobs=1).fit(X_train,y_train)
print(f"Correct-direction constrained AUC={roc_auc_score(y_test,good.predict_proba(X_test)[:,1]):.3f}; wrong-direction AUC={roc_auc_score(y_test,bad.predict_proba(X_test)[:,1]):.3f}")

**Use this when…** structured tabular data contain nonlinearities and interactions and predictive performance is primary.

        **Don't use this when…** you need smooth extrapolation, a simple scorecard, or constraints you cannot state correctly.

        ## Try this

        1. Halve learning rate and double rounds; compare validation minima.
2. Raise tree depth and watch train loss improve faster than validation.
3. Apply monotonic constraints only where domain knowledge is defensible.